# Solar Filament Segmentation 2026 — full pipeline (ten-model DINOv3 ensemble + learned selector "C")

This notebook runs the whole submitted pipeline from the released weights to a submission CSV, and shows the predicted filaments on the H-alpha images.

| Stage | What happens | Where |
| --- | --- | --- |
| 0 | fetch the ten weight files and verify SHA-256 | `scripts/fetch_weights.py` |
| 1 | read each test image (8-bit grayscale, native 2048×2048, scaled to [0,1]) | `filament.data.read_gray` |
| 2 | DINOv3 ViT-L/16 + high-resolution decoder; overlapping 512 px tiles, identity/h-flip/v-flip views; five fold models per seed averaged | `filament.engine.predict` |
| 3 | average the two seed-level probability maps (ten-model mean, float32) | `scripts/reinfer_10m.py` |
| 4 | selector C: components of p > 0.4 (area ≥ 100 px), two gradient-boosted regressors on 11 label-free descriptors, keep score ≥ 0.25 | `scripts/candidate_select.py`, `scripts/apply_selector.py` |
| 5 | disjoint instances → compressed COCO RLE CSV | `filament.submission` |

Environment variables (all optional): `EMA10_N_IMAGES` (0 = all 180 images; e.g. 3 for a quick check), `EMA10_IMAGES_DIR` (default `data/test_images`), `EMA10_OUT_DIR`, `EMA10_DEVICE` (`auto`/`cuda`/`mps`/`cpu`), `EMA10_ABORT_SINGLE_SECONDS` / `EMA10_ABORT_MEAN_SECONDS` (safety stops, GPU defaults 90 s / 25 s; raise them on slower devices), and for the optional sections `EMA10_TRAIN` (`plan`/`run`), `EMA10_TRAIN_FOLD`, `EMA10_EVAL_N`.
Expected cost on one RTX 3090: about 98 minutes for all 180 images (two five-model passes of about 16 s per image), 7.6 GB GPU memory per pass.

In [ ]:
import os, subprocess, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "weights_manifest_10m.json").exists() and ROOT != ROOT.parent: ROOT = ROOT.parent   # repository root
assert (ROOT / "weights_manifest_10m.json").exists(), "run the notebook from inside the repository"
os.chdir(ROOT)
N = int(os.environ.get("EMA10_N_IMAGES", "0")); OUT = Path(os.environ.get("EMA10_OUT_DIR", "out_ens10")); IMAGES = os.environ.get("EMA10_IMAGES_DIR", "data/test_images"); DEVICE = os.environ.get("EMA10_DEVICE", "auto")
print("repository:", ROOT, "| images:", IMAGES, "| n_images:", N or "all", "| out:", OUT)

In [ ]:
# Optional: install the pinned dependencies (skip if the environment is ready): EMA10_PIP=1
if os.environ.get("EMA10_PIP") == "1":
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], capture_output=True, text=True); print(r.stdout[-800:], r.stderr[-800:]); assert r.returncode == 0

## Stage 0 — weights (downloaded from the Hugging Face Hub, SHA-256 verified; nothing is trained here)

In [ ]:
r = subprocess.run([sys.executable, "scripts/fetch_weights.py"], capture_output=True, text=True); print(r.stdout[-1500:]); print(r.stderr[-800:]); assert r.returncode == 0

## Stages 1–5 — inference, ten-model mean, selector C, CSV

In [ ]:
cmd = [sys.executable, "scripts/reinfer_10m.py", "--bundle", ".", "--repo", ".", "--out-dir", str(OUT), "--images-dir", IMAGES, "--ids-file", "test_ids.txt", "--post", "A2/post.json", "--device", DEVICE]
if N: cmd += ["--n", str(N)]
# safety stops tuned for a GPU (abort if one image takes > 90 s or the first-10 mean > 25 s); raise them on slow devices, e.g. EMA10_ABORT_SINGLE_SECONDS=100000 EMA10_ABORT_MEAN_SECONDS=100000
if os.environ.get("EMA10_ABORT_SINGLE_SECONDS"): cmd += ["--abort-single-seconds", os.environ["EMA10_ABORT_SINGLE_SECONDS"]]
if os.environ.get("EMA10_ABORT_MEAN_SECONDS"): cmd += ["--abort-mean-seconds", os.environ["EMA10_ABORT_MEAN_SECONDS"]]
print(" ".join(cmd[1:]))
r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout[-3500:]); print(r.stderr[-1500:]); assert r.returncode == 0

## Result check
`C_10m.csv` is the submitted-candidate CSV (selector C). On the same kind of environment as the original run (RTX 3090, CUDA 12.8, torch 2.8.0) the probabilities are bit-identical to the stored reference and the CSV is byte-identical to `submission_C_10m.csv`. On other devices small differences are expected; the comparison below then reports instance matching.

In [ ]:
print((OUT / "COMPARISON_C_10m.md").read_text())
print(json.dumps(json.loads((OUT / "probs_sha_comparison.json").read_text()), ensure_ascii=False)[:700])

## Predicted morphology on H-alpha images (selector C; instances in colour; test ground truth is not available)

In [ ]:
import csv, numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from pycocotools import mask as mu
sys.path.insert(0, "src")
from filament.data import read_gray
rows = {}
with open(OUT / "C_10m.csv") as f:
    for r_ in csv.DictReader(f): rows.setdefault(r_["filament_id"].rsplit("_", 1)[0], []).append(r_["segmentation_rle"])
ids = [l.strip() for l in open(OUT / "ids.txt") if l.strip()]
show = [s for s in ids if rows.get(s)][:3]
def outline(m, w=3): return m & ~ndi.binary_erosion(m, iterations=w)
colors = [(1.0, 0.25, 0.1), (0.1, 0.8, 1.0), (1.0, 0.9, 0.1), (0.6, 1.0, 0.2)]
fig, ax = plt.subplots(len(show), 3, figsize=(15, 5 * len(show)), squeeze=False)
for i, s in enumerate(show):
    img = read_gray(Path(IMAGES) / f"{s}.jpeg"); ms = [mu.decode({"size": [2048, 2048], "counts": c.encode()}).astype(bool) for c in rows[s]]
    big = np.stack([img] * 3, -1).astype(np.float32) / 255
    for k, m in enumerate(ms): big[ndi.binary_dilation(outline(m, 2), iterations=2)] = colors[k % 4]
    ax[i, 0].imshow(big); ax[i, 0].set_title(f"{s}: {len(ms)} predicted instances (outlines)"); ax[i, 0].axis("off")
    for j, m in enumerate(sorted(ms, key=lambda x: -x.sum())[:2]):
        ys, xs = np.nonzero(m); pad = 70; y0, y1, x0, x1 = max(ys.min() - pad, 0), min(ys.max() + pad, 2048), max(xs.min() - pad, 0), min(xs.max() + pad, 2048)
        crop = np.stack([img[y0:y1, x0:x1]] * 3, -1).astype(np.float32) / 255; crop[outline(m, 1)[y0:y1, x0:x1]] = colors[0]
        ax[i, 1 + j].imshow(crop); ax[i, 1 + j].set_title(f"zoom on instance {j + 1} ({int(m.sum())} px)"); ax[i, 1 + j].axis("off")
    for j in range(len(ms), 2): ax[i, 1 + j].axis("off")
plt.tight_layout(); plt.savefig(OUT / "predictions_preview.png", dpi=100); plt.show()

## Optional A — training (off by default: the notebook loads the released weights)
The organizers ask that the notebook contain all code needed to reproduce the work, with an optional section to train instead of loading the released weights. Training is **not** required to reproduce the predictions.
Each of the ten models is one fold of one seed: the DINOv3 ViT-L/16 model is fine-tuned on that fold's training images (`training/` holds the split, nested split, and the inner-selection records with the number of updates), with EMA 0.999. `scripts/train_fold.py` re-runs one fold; by default it only checks that the split/selection hashes match the released models and prints the plan (`EMA10_TRAIN=run` actually trains: needs a GPU, the competition training data under `data/raw/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026`, about 1.2–2 h per fold). A retrained model will not be bit-identical to the released one (the original training commit is unknown and GPU training is non-deterministic).

In [ ]:
TRAIN = os.environ.get("EMA10_TRAIN", "")            # "" = skip, "plan" = check hashes and print the plan, "run" = train one fold (GPU)
FOLD = os.environ.get("EMA10_TRAIN_FOLD", "0")
if TRAIN in ("plan", "run"):
    DATA = "data/raw/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"
    cmd = [sys.executable, "scripts/train_fold.py", "--fold", FOLD] + (["--data-root", DATA] if Path(DATA).exists() else []) + (["--run"] if TRAIN == "run" else [])
    r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout[-3000:]); print(r.stderr[-1500:])
else:
    print("training skipped (set EMA10_TRAIN=plan or run)")

## Optional B — evaluation protocol on held-out training images
Scoring follows the public self-evaluation rule: for every annotation set a prediction matches a ground-truth instance when IoU > 0.5; matched IoUs, TP, FP, FN are summed over annotation sets and images, PQ = ΣIoU / (TP + FP/2 + FN/2) (`src/filament/metrics.py`).
This cell illustrates the protocol on training images that **one fold model never saw** (fold-0 seed-42 model, fixed rule A2). It is not the score of the submitted ensemble. Needs the competition training data under `data/raw/...` (set `EMA10_EVAL_N` to the number of images, e.g. 3).

In [ ]:
EVAL_N = int(os.environ.get("EMA10_EVAL_N", "0"))
if EVAL_N:
    import numpy as np, torch
    sys.path.insert(0, "src")
    from filament import engine, native_infer
    from filament.data import read_gray, load_train, ROOT
    from filament.metrics import PQStats, encode, pq_stats
    ns = json.load(open("training/nested_split.json"))["folds"][0]; stems = sorted(ns["outer_eval"])[:EVAL_N]
    dev = torch.device("cuda" if torch.cuda.is_available() and DEVICE in ("auto", "cuda") else ("mps" if torch.backends.mps.is_available() and DEVICE in ("auto", "mps") else "cpu"))
    models, cfg, mans = native_infer.load_weights([Path("weights/fold0/weights_ema_0.999.pt")], dev)
    assert set(stems).isdisjoint(mans[0]["fit"]), "evaluation images must not be training images of this model"
    names = {p.stem: n for n, p in load_train().items()}; post = native_infer.load_post(Path("A2/post.json")); gts = engine.gt_rles(stems); tot = PQStats()
    for s in stems:
        prob = engine.predict(models, read_gray(ROOT / "train/train_images" / names[s]), dev, cfg); pred = encode(native_infer.masks_from_prob(prob, post))
        for g in gts[s]: tot.add(pq_stats(g, pred))
    print(f"{len(stems)} held-out images, fold-0 single model + rule A2: PQ {tot.pq:.4f} (TP {tot.tp}, FP {tot.fp}, FN {tot.fn}, SQ {tot.sq:.3f}, RQ {tot.rq:.3f})")
else:
    print("evaluation skipped (set EMA10_EVAL_N)")

## Notes
- The reference CSV `submission_C_10m.csv` is what was submitted to Kaggle (submission 58204294, public score 0.38). `submission_A2_10m.csv` is the fixed-rule alternative (not submitted).
- Weights and the selector are checked against SHA-256 values before use. The seed-43 fold-3 weight manifest was reconstructed (see `PROVENANCE.md`).
- No competition data is bundled; place the test JPEGs under `data/test_images/`.